Parameters and Profile

In [0]:
import random
from collections import Counter

CATALOG = "dbw_taxrisk_dev"
COUNTRY_CODE = "CO"
N_CREATORS = 200
RISKY_SHARE = 0.25
SEED = 42

random.seed(SEED)

profile = (spark.table(f"{CATALOG}.config.country_profile")
           .filter(f"country_code = '{COUNTRY_CODE}'")
           .first())
print(profile.country_name_en, profile.currency_code, profile.languages, profile.disclosure_lexicon)

Venues: Prices are invented synthetic values

In [0]:
CITIES = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Cartagena"]
SUFFIXES = ["del Norte", "Verde", "Dorado", "Central", "Antiguo", "Real", "Azul"]
CATEGORIES = {
    "restaurante": (["La Mesa", "Sabor", "Fogón", "Casa", "Huerta"], 120_000),
    "café":        (["Grano", "Altura", "Taza", "Tostadero", "Origen"], 30_000),
    "bar":         (["Barril", "Terraza", "Coctel", "Noche", "Esquina"], 150_000),
    "hotel":       (["Hacienda", "Boutique", "Mirador", "Plaza", "Reserva"], 600_000),
    "spa":         (["Armonía", "Calma", "Bruma", "Oasis", "Pétalo"], 300_000),
}
venues = []
for i in range(60):
    cat = random.choice(list(CATEGORIES))
    words, base = CATEGORIES[cat]
    venues.append({
        "venue_id": f"V{i:03d}",
        "name": f"{random.choice(words)} {random.choice(SUFFIXES)}",
        "category": cat,
        "city": random.choice(CITIES),
        "typical_ticket_cop": int(base * random.uniform(0.7, 1.5)),
    })

Creators

In [0]:
FIRST = ["sofia", "camila", "valentina", "juan", "andres", "laura",
         "daniela", "mateo", "santiago", "luisa", "paula", "felipe"]
THEMES = ["viaja", "come", "vive", "style", "fit", "foodie", "trip", "moda"]
TIERS = {  # tier: (min followers, max followers, annual declared income range in COP)
    "nano":  (1_000, 10_000, (6_000_000, 24_000_000)),
    "micro": (10_000, 100_000, (24_000_000, 90_000_000)),
    "macro": (100_000, 1_000_000, (90_000_000, 400_000_000)),
}
risky_idx = set(random.sample(range(N_CREATORS), int(N_CREATORS * RISKY_SHARE)))
creators, used = [], set()
for i in range(N_CREATORS):
    tier = random.choices(list(TIERS), weights=[60, 32, 8])[0]
    f_lo, f_hi, (inc_lo, inc_hi) = TIERS[tier]
    while True:
        handle = f"@{random.choice(FIRST)}_{random.choice(THEMES)}{random.randint(10, 99)}"
        if handle not in used:
            used.add(handle)
            break
    creators.append({
        "creator_id": f"C{i:04d}",
        "handle": handle,
        "tier": tier,
        "followers": random.randint(f_lo, f_hi),
        "declared_income_cop": int(random.uniform(inc_lo, inc_hi)),
        "is_registered_taxpayer": random.random() < 0.9,
        "is_risky": i in risky_idx,
    })

Sanity check

In [0]:
print(len(venues), "venues;", len(creators), "creators")
print(Counter(c["tier"] for c in creators))
print("risky creators:", sum(c["is_risky"] for c in creators))
print("unique handles:", len({c["handle"] for c in creators}))
print(creators[0]); print(venues[0])

Templates

In [0]:
import re
from datetime import datetime, timedelta

random.seed(SEED + 1)  # each stage reseeds, so it can be re-run on its own

LEX = profile.disclosure_lexicon
PAID_TAGS = [t for t in LEX if t in ("#publicidad", "#colaboración")]
INKIND_TAGS = [t for t in LEX if t in ("#canje", "#regalo", "#colaboración")]

T_PAID = [
    "Colaboración con {venue} en {city}. Gracias por confiar en mí. {tag}",
    "Anuncio: {venue}, {city}. Los espero. {tag}",
]
T_INKIND_DISCLOSED = [
    "Cortesía de {venue}: una experiencia increíble en {city}. {tag}",
    "Gracias {venue} por la invitación. {tag}",
]
T_PRAISE = [  # used by BOTH undisclosed in-kind posts and ambiguous organic posts
    "Qué noche en {venue}, {city}. Todo perfecto, gracias equipo!",
    "Domingo especial en {venue}. Lo recomiendo muchísimo",
    "Así se disfruta {city}: {venue} no falla",
]
T_ORGANIC = [
    "Planes de {category} en {city} este fin de semana",
    "Mi rutina de la semana en {city}",
    "Probando algo nuevo hoy, ya les cuento",
]
IMG = {
    "restaurante": "mesa con platos y copas",
    "café": "taza de café y postre en una mesa",
    "bar": "cócteles sobre la barra",
    "hotel": "habitación con vista a la ciudad",
    "spa": "bandeja con productos de spa",
}
IMG_CUE = "tarjeta de bienvenida del local sobre la mesa"
PLATFORMS = ["instagram", "tiktok", "youtube"]
YEAR_START = datetime(2025, 1, 1)

Post builder

In [0]:
def make_post(c, n):
    v = random.choice(venues)
    fill = dict(venue=v["name"], city=v["city"], category=v["category"])
    weights = [55, 10, 35] if c["is_risky"] else [82, 15, 3]  # organic, paid, in_kind
    cls = random.choices(["organic", "paid", "in_kind"], weights)[0]
    venue_id, value, disclosed, ambiguous = v["venue_id"], None, False, False
    image = IMG[v["category"]]

    if cls == "paid":
        caption = random.choice(T_PAID).format(tag=random.choice(PAID_TAGS), **fill)
        disclosed = True
    elif cls == "in_kind":
        mult = random.choice([1, 2]) if v["category"] == "hotel" else random.choice([1, 2, 2, 3])
        value = int(v["typical_ticket_cop"] * mult * random.uniform(0.9, 1.2))
        disclosed = random.random() < (0.6 if c["is_risky"] else 0.9)
        if disclosed:
            caption = random.choice(T_INKIND_DISCLOSED).format(tag=random.choice(INKIND_TAGS), **fill)
        else:
            caption = random.choice(T_PRAISE).format(**fill)
        if random.random() < 0.3:
            image += "; " + IMG_CUE
    else:  # organic
        if random.random() < 0.2:
            ambiguous = True  # looks like an in-kind post, but the creator paid
            caption = random.choice(T_PRAISE).format(**fill)
        else:
            caption = random.choice(T_ORGANIC).format(**fill)
            venue_id, image = None, "foto de la ciudad"

    when = YEAR_START + timedelta(days=random.randint(0, 364), seconds=random.randint(0, 86399))
    post = {
        "post_id": f"P{n:06d}",
        "platform": random.choices(PLATFORMS, [60, 30, 10])[0],
        "creator_handle": c["handle"],
        "posted_at": when.isoformat(),
        "caption": caption,
        "hashtags": re.findall(r"#\w+", caption),
        "image_description": image,
        "likes": int(c["followers"] * random.uniform(0.01, 0.08)),
    }
    label = {
        "post_id": post["post_id"],
        "creator_id": c["creator_id"],
        "true_class": cls,
        "venue_id": venue_id,
        "true_value_cop": value,
        "is_disclosed": disclosed,
        "is_ambiguous": ambiguous,
    }
    return post, label

Generate and check

In [0]:
posts, labels, n = [], [], 0
for c in creators:
    for _ in range(random.randint(15, 35)):
        p, l = make_post(c, n)
        posts.append(p); labels.append(l); n += 1

risky = {c["creator_id"]: c["is_risky"] for c in creators}
print(len(posts), "posts")
print(Counter(l["true_class"] for l in labels))
print("in-kind posts, risky vs benign creators:",
      Counter(risky[l["creator_id"]] for l in labels if l["true_class"] == "in_kind"))
ik = [l for l in labels if l["true_class"] == "in_kind"]
print("in-kind posts without disclosure:", sum(not l["is_disclosed"] for l in ik), "of", len(ik))
print("organic posts that look like in-kind:", sum(l["is_ambiguous"] for l in labels))
print("total in-kind value (COP):", sum(l["true_value_cop"] for l in ik))
for cls in ["paid", "in_kind", "organic"]:
    i = next(i for i, l in enumerate(labels) if l["true_class"] == cls)
    print(cls, "->", posts[i]["caption"], "|", posts[i]["image_description"])

Write the files

In [0]:
import json, os

LANDING = f"/Volumes/{CATALOG}/bronze/landing"
BATCH = "batch_001"

def write_jsonl(rows, folder, filename):
    """Write a list of dicts as JSON-lines into LANDING/folder/filename."""
    path = f"{LANDING}/{folder}"
    os.makedirs(path, exist_ok=True)
    full = f"{path}/{filename}"
    with open(full, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    return full, len(rows)

# What the pipeline is allowed to see
registry = [{k: v for k, v in c.items() if k != "is_risky"} for c in creators]
truth_creators = [{"creator_id": c["creator_id"], "is_risky": c["is_risky"]} for c in creators]

outputs = [
    write_jsonl(posts,          "posts",        f"posts_{BATCH}.jsonl"),
    write_jsonl(venues,         "venues",       f"venues_{BATCH}.jsonl"),
    write_jsonl(registry,       "registry",     f"registry_{BATCH}.jsonl"),
    # Hidden truth, evaluation only
    write_jsonl(labels,         "ground_truth", f"labels_{BATCH}.jsonl"),
    write_jsonl(truth_creators, "ground_truth", f"creators_truth_{BATCH}.jsonl"),
]
for p, n in outputs:
    print(n, "rows ->", p)

Verify

In [0]:
for folder in ["posts", "venues", "registry", "ground_truth"]:
    for f in dbutils.fs.ls(f"{LANDING}/{folder}"):
        print(f"{folder:13} {f.name:32} {f.size/1024:8.1f} KB")

print("\nregistry columns:", list(registry[0].keys()))
print("is_risky in registry?", "is_risky" in registry[0])

# First 2 lines of the posts file, to see exactly what Bronze will receive
with open(f"{LANDING}/posts/posts_{BATCH}.jsonl", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline().strip())